In [15]:
# add input file and reference genome here
input_file = "test_variants.txt"
genome_build = "hg19"   # options: hg19 / hg38

# do not change below unless clinvar file needs an update
sleep_time = 7.0        # API rate limit safety
clinvar_file = "variant_summary.txt"

# optional:
# change this to True to download the latest ClinVar variant summary file from NCBI
download_clinvar = False
# NCBI FTP URL for the latest ClinVar variant summary file (tab-delimited, gzipped)
clinvar_download_url = "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz"

In [16]:
# libraries
import pandas as pd
import requests
import time
import os
import gzip
import shutil

In [17]:
# CLINVAR DOWNLOAD
if download_clinvar:
    print("Downloading latest ClinVar variant summary...")

    gz_file = "variant_summary.txt.gz"
    out_file = "variant_summary.txt"

    # download
    with requests.get(clinvar_download_url, stream=True) as r:
        r.raise_for_status()
        with open(gz_file, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)

    print("Download complete. Unzipping...")

    # unzip
    with gzip.open(gz_file, "rb") as f_in:
        with open(out_file, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)

    print(f"ClinVar file ready: {out_file}")

    # override input file to use downloaded version
    clinvar_file = out_file

In [18]:
# only pull necessary columns
cols_to_use = [
    "Chromosome",
    "PositionVCF",
    "ReferenceAlleleVCF",
    "AlternateAlleleVCF",
    "ClinicalSignificance",
    "ReviewStatus",
    "NumberSubmitters"
]

clinvar_df = pd.read_csv(
    clinvar_file,
    sep="\t",
    usecols=cols_to_use,
    dtype=str,
    low_memory=True,
    engine="c"
)

clinvar_df.columns = clinvar_df.columns.str.strip().str.replace("#", "")


# convert to variant id format
clinvar_df["VariantID"] = (
    clinvar_df["Chromosome"].astype(str) + "-" +
    clinvar_df["PositionVCF"].astype(str) + "-" +
    clinvar_df["ReferenceAlleleVCF"].astype(str) + "-" +
    clinvar_df["AlternateAlleleVCF"].astype(str)
)


# subset
clinvar_subset = clinvar_df[
    ["VariantID", "ClinicalSignificance", "ReviewStatus", "NumberSubmitters"]
].dropna()

In [19]:
# process input file
input_df = pd.read_csv(input_file, dtype=str)

print("Input columns:", input_df.columns.tolist())

# Case 1: Proper header exists
if "Variant ID" in input_df.columns:
    input_df = input_df.rename(columns={"Variant ID": "VariantID"})

# Case 2: Already named correctly
elif "VariantID" in input_df.columns:
    pass

# Case 3: No header (first row is actually data)
else:
    print("No 'Variant ID' column found. Assuming single-column file without header.")
    
    input_df = pd.read_csv(input_file, dtype=str, header=None)
    input_df.columns = ["VariantID"]

# Clean values
input_df["VariantID"] = (
    input_df["VariantID"]
    .astype(str)
    .str.replace('"', '', regex=False)
    .str.strip()
)

# Drop empty rows just in case
input_df = input_df[input_df["VariantID"] != ""]

Input columns: ['Variant ID']


In [20]:
# merge
annotation_df = input_df.merge(
    clinvar_subset,
    on="VariantID",
    how="left"
)

In [21]:
# reference selection
genome_build_clean = genome_build.lower().strip()

if genome_build_clean in ["hg19", "grch37"]:
    dataset = "gnomad_r2_1"

elif genome_build_clean in ["hg38", "grch38"]:
    dataset = "gnomad_r3"

else:
    raise ValueError(
        f"Invalid genome_build: '{genome_build}'. "
        "Use 'hg19', 'GRCh37', 'hg38', or 'GRCh38'."
    )

In [22]:
# API functions
def fetch_block(variant_id, dataset, block, retries=3):
    url = "https://gnomad.broadinstitute.org/api"

    query = {
        "query": f"""
        {{
          variant(variantId: "{variant_id}", dataset: {dataset}) {{
            {block} {{
              af
              ac
              an
            }}
          }}
        }}
        """
    }

    for _ in range(retries):
        try:
            r = requests.post(url, json=query)
            data = r.json()

            if "errors" in data:
                return None

            variant = data.get("data", {}).get("variant")
            if not variant:
                return None

            return variant.get(block)

        except:
            time.sleep(0.5)

    return None


def fetch_af(variant_id, dataset):
    genome = fetch_block(variant_id, dataset, "genome")
    exome = fetch_block(variant_id, dataset, "exome")

    genome_af = genome.get("af") if genome else None
    exome_af = exome.get("af") if exome else None

    genome_ac = genome.get("ac") if genome else 0
    genome_an = genome.get("an") if genome else 0

    exome_ac = exome.get("ac") if exome else 0
    exome_an = exome.get("an") if exome else 0

    total_ac = (genome_ac or 0) + (exome_ac or 0)
    total_an = (genome_an or 0) + (exome_an or 0)

    total_af = total_ac / total_an if total_an else None

    return {
        "genome_af": genome_af,
        "exome_af": exome_af,
        "total_af": total_af
    }


In [23]:
# annotation
af_list = []

for vid in annotation_df["VariantID"]:
    af = fetch_af(vid, dataset)
    af_list.append(af)

    time.sleep(sleep_time)  # respect API limit


# Convert list of dicts → DataFrame
af_df = pd.DataFrame(af_list)

# Merge with original dataframe
annotation_df = pd.concat(
    [annotation_df.reset_index(drop=True), af_df],
    axis=1
)

In [24]:
# create output filename based on input
base_name = os.path.splitext(os.path.basename(input_file))[0]
output_file = f"{base_name}_results.csv"

annotation_df.to_csv(output_file, index=False)

print(f"Saved results to: {output_file}")

Saved results to: test_variants_results.csv
